In [1]:
from pathlib import Path
import csv
import json
import math
import re
import statistics
from collections import defaultdict

# Shared definitions intentionally included inside each notebook.
# No external Python helper file is needed.
DATA_FOLDER = Path.cwd() / "csv_data"
CONDITIONS = ["Live", "20 mg/L", "320 mg/L", "Dead"]
LABELS = {"live": "Live", "20": "20 mg/L", "320": "320 mg/L", "dead": "Dead"}
FOLDER_RE = re.compile(r"^(Live|20|320|Dead)_cell(\d+)_crop(\d+)$", re.I)
LIMIT_RMS_PN = 25.0
MIN_LINEAR_NM = 5.0
MIN_ELASTICITY_NM = 10.0
MAX_OVERLAP_NM = 5.0


def read_csv(path):
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def num(value):
    try:
        result = float(value)
        return result if math.isfinite(result) else None
    except (ValueError, TypeError, OverflowError):
        return None


def identity(row):
    return (row.get("filename", ""), row.get("curve_number", ""),
            row.get("position_index", ""))


def interval(row, prefix):
    a = num(row.get(prefix + "_x_min_used_nm"))
    b = num(row.get(prefix + "_x_max_used_nm"))
    return tuple(sorted((a, b))) if a is not None and b is not None else None


def widths(row):
    linear = interval(row, "linear")
    elastic = interval(row, "elasticity")
    linear_nm = linear[1] - linear[0] if linear else None
    elasticity_nm = elastic[1] - elastic[0] if elastic else None
    # Indentation-side part of the fitted elasticity interval, relative to
    # the processed curve's contact at X = 0 nm, clipped to the fit interval.
    contact_nm = (max(0.0, min(0.0, elastic[1]) - elastic[0])
                  if elastic else None)
    overlap_nm = (max(0.0, min(linear[1], elastic[1]) - max(linear[0], elastic[0]))
                  if linear and elastic else None)
    xc = num(row.get("contact_point_x_nm"))
    fitted_contact_nm = (max(0.0, min(max(xc, elastic[0]), elastic[1]) - elastic[0])
                         if elastic and xc is not None else None)
    return linear_nm, elasticity_nm, contact_nm, fitted_contact_nm, overlap_nm


def trace(points, y_column, used_column=None):
    selected = []
    for row in points:
        if used_column and row.get(used_column) != "1":
            continue
        x, y = num(row.get("x_nm")), num(row.get(y_column))
        if x is not None and y is not None:
            selected.append([x, y])
    if used_column:
        selected.sort(key=lambda point: point[0])
    return selected


def elasticity_rms_from_points(points):
    """Recalculate custom RMS from actual plotted fitted points, in pN.

    Force values in the point CSV are nN, so multiply RMS by 1000.
    N is the number of valid pairs in the elasticity fitting interval.
    The three fitted parameters are E, contact position and baseline.
    """
    residual_sq = []
    for point in points:
        if point.get("elasticity_fit_used") != "1":
            continue
        measured = num(point.get("measured_y_nN"))
        predicted = num(point.get("elasticity_fit_y_nN"))
        if measured is not None and predicted is not None:
            residual_sq.append((measured - predicted) ** 2)
    n = len(residual_sq)
    if n <= 3:
        return None, n
    return math.sqrt(math.fsum(residual_sq) / (n - 3)) * 1000.0, n


def derived_values(summary, points):
    if summary is None:
        return {}
    v = dict(summary)
    slope = num(v.get("linear_slope_bruker_N_per_m"))
    linear_nm, elasticity_nm, contact_nm, fitted_nm, overlap_nm = widths(v)
    v["bacterial_spring_constant_nN_per_um"] = abs(slope) * 1000 if slope is not None else None
    v["linear_fit_range_nm"] = linear_nm
    v["elasticity_fit_range_nm"] = elasticity_nm
    v["elasticity_fit_range_to_preprocessing_contact_nm"] = contact_nm
    v["elasticity_fit_range_to_fitted_contact_nm"] = fitted_nm
    v["fit_overlap_nm"] = overlap_nm
    # Recompute directly instead of multiplying the old RMS a second time.
    # Correct for both old N and newer N-3 Jython exports, including 320 Cell 3.
    v["elasticity_rmsd_custom_pN"], v["elasticity_recalculated_T"] = elasticity_rms_from_points(points)
    jpk = num(v.get("elasticity_rms_bruker_pN"))
    custom = v["elasticity_rmsd_custom_pN"]
    v["elasticity_rms_difference_pN"] = (custom - jpk if custom is not None and jpk is not None else None)
    if custom is not None and jpk is not None:
        avg = (abs(custom) + abs(jpk)) / 2
        v["elasticity_rms_percentage_difference"] = abs(custom - jpk) / avg * 100 if avg else 0.0
    else:
        v["elasticity_rms_percentage_difference"] = None
    return v


def rejection_reasons(v):
    if not v:
        return ["No uniquely matching summary row"]
    checks = [
        (num(v.get("linear_rmsd_custom_pN")), "Linear RMS > 25 pN or missing", "max", LIMIT_RMS_PN),
        (num(v.get("elasticity_rms_bruker_pN")), "JPK elasticity RMS > 25 pN or missing", "max", LIMIT_RMS_PN),
        (v.get("linear_fit_range_nm"), "Linear fitting width < 5 nm or missing", "min", MIN_LINEAR_NM),
        (v.get("elasticity_fit_range_to_preprocessing_contact_nm"), "Elasticity indentation width < 10 nm or missing", "min", MIN_ELASTICITY_NM),
        (v.get("fit_overlap_nm"), "Fitting interval overlap > 5 nm or missing", "max", MAX_OVERLAP_NM),
    ]
    reasons = []
    for value, reason, direction, limit in checks:
        if value is None or (direction == "max" and value > limit) or (direction == "min" and value < limit):
            reasons.append(reason)
    return reasons


def curve_inventory():
    """Read point CSVs first, match summaries by full filename/curve/position.

    If historic curve_number in a point file is wrong, recover by a unique
    filename + position match. Summary-only leftovers are never included.
    """
    if not DATA_FOLDER.is_dir():
        raise FileNotFoundError("Run this notebook from the repository root containing csv_data")
    folders = [p for p in DATA_FOLDER.iterdir() if p.is_dir() and FOLDER_RE.fullmatch(p.name)]
    folders.sort(key=lambda p: (CONDITIONS.index(LABELS[FOLDER_RE.fullmatch(p.name).group(1).lower()]),
                                int(FOLDER_RE.fullmatch(p.name).group(2)),
                                int(FOLDER_RE.fullmatch(p.name).group(3))))
    for folder in folders:
        matcher = FOLDER_RE.fullmatch(folder.name)
        condition = LABELS[matcher.group(1).lower()]
        cell, crop = int(matcher.group(2)), int(matcher.group(3))
        paths = sorted(folder.glob("AFM_fit_points_curve_*.csv"))
        if not paths:
            continue
        summary_file = folder / "AFM_jython_summary.csv"
        summaries = read_csv(summary_file) if summary_file.exists() else []
        by_key, by_fallback = defaultdict(list), defaultdict(list)
        for row in summaries:
            if row.get("filename") and row.get("position_index"):
                by_key[identity(row)].append(row)
                by_fallback[(row["filename"], row["position_index"])].append(row)
        counted = set()
        for path in paths:
            points = read_csv(path)
            if not points:
                continue
            first = points[0]
            matches = by_key.get(identity(first), [])
            if not matches:
                matches = by_fallback.get((first.get("filename", ""), first.get("position_index", "")), [])
            summary = matches[0] if len(matches) == 1 else None
            key = identity(summary) if summary else None
            duplicate = key in counted if key else False
            if key:
                counted.add(key)
            values = derived_values(summary, points)
            reasons = rejection_reasons(values)
            if duplicate:
                reasons.append("Duplicate individual fit-point export")
            measured = trace(points, "measured_y_nN")
            linear = trace(points, "linear_fit_y_nN", "linear_fit_used")
            elastic = trace(points, "elasticity_fit_y_nN", "elasticity_fit_used")
            if not (measured and linear and elastic):
                reasons.append("Missing measured or fitted plotting points")
            yield {
                "condition": condition, "cell": cell, "crop": crop, "folder": folder.name,
                "curve": (summary.get("curve_number") if summary else first.get("curve_number")),
                "position": first.get("position_index"), "filename": first.get("filename", ""),
                "values": values, "reasons": reasons, "accepted": not reasons,
                "measured": measured, "linear": linear, "elasticity": elastic,
            }


def med(values):
    return statistics.median(values) if values else None


def mad(values):
    if not values:
        return None
    centre = statistics.median(values)
    return statistics.median(abs(v - centre) for v in values)


def grouping(curves):
    groups = defaultdict(list)
    for curve in curves:
        if curve["accepted"]:
            groups[(curve["condition"], curve["cell"])].append(curve)
    return groups


def cell_statistics(curves):
    result = {}
    for key, group in grouping(curves).items():
        kb = [num(c["values"].get("bacterial_spring_constant_nN_per_um")) for c in group]
        young = [num(c["values"].get("youngs_modulus_MPa")) for c in group]
        kb, young = [x for x in kb if x is not None], [x for x in young if x is not None]
        result[key] = {
            "n": len(group), "kb_median": med(kb), "kb_mad": mad(kb),
            "E_median": med(young), "E_mad": mad(young)
        }
    return result


def condition_statistics(cells):
    result = {}
    for cond in CONDITIONS:
        rows = [v for (c, _), v in sorted(cells.items(), key=lambda t: t[0][1]) if c == cond]
        def aggregate(key):
            v = [r[key] for r in rows if r[key] is not None]
            return (statistics.mean(v) if v else None,
                    statistics.stdev(v) if len(v) >= 2 else None)
        result[cond] = (aggregate("kb_median"), aggregate("E_median"))
    return result

from collections import Counter
import zipfile

HEADERS = [
    'Condition','Cell','Crop','Source folder','Curve number','Position index',
    'Linear slope (N/m)','Bacterial spring constant (nN/µm)',
    'Linear RMSD custom (pN)','Linear X minimum (nm)','Linear X maximum (nm)',
    'Linear fit range (nm)','Linear/elasticity fit overlap (nm)',
    "Young's modulus (MPa)",'JPK Elasticity RMS (pN)',
    'Elasticity RMSD custom (pN)','Elasticity RMS percentage difference (%)',
    'Elasticity X minimum (nm)','Elasticity X maximum (nm)',
    'Elasticity fit range (nm)','Elasticity range to preprocessing contact (nm)',
    'Elasticity range to fitted contact (nm)','Fitted contact X (nm)',
    'Fitted contact Y (nN)','Source filename'
]
COLUMNS = [
    None,None,None,None,None,None,
    'linear_slope_bruker_N_per_m','bacterial_spring_constant_nN_per_um',
    'linear_rmsd_custom_pN','linear_x_min_used_nm','linear_x_max_used_nm',
    'linear_fit_range_nm','fit_overlap_nm',
    'youngs_modulus_MPa','elasticity_rms_bruker_pN',
    'elasticity_rmsd_custom_pN','elasticity_rms_percentage_difference',
    'elasticity_x_min_used_nm','elasticity_x_max_used_nm',
    'elasticity_fit_range_nm','elasticity_fit_range_to_preprocessing_contact_nm',
    'elasticity_fit_range_to_fitted_contact_nm','contact_point_x_nm',
    'contact_point_y_fit_nN',None
]
CSV_NAMES = {'Live':'Live.csv','20 mg/L':'20_mg_L.csv',
             '320 mg/L':'320_mg_L.csv','Dead':'Dead.csv'}
SHEET_NAMES = {'Live':'Live','20 mg/L':'20 mg_L',
               '320 mg/L':'320 mg_L','Dead':'Dead'} # Excel forbids / in tab names

def export_text(value):
    if value is None or value == '':
        return ''
    return format(value,'.17g') if isinstance(value,float) else str(value)

def export_row(c):
    v = c['values']
    items = [c['condition'],c['cell'],c['crop'],c['folder'],c['curve'],c['position']]
    items.extend(export_text(v.get(col)) for col in COLUMNS[6:24])
    items.append(c['filename'])
    assert len(items) == len(HEADERS)
    return items

curves = list(curve_inventory())
accepted = [c for c in curves if c['accepted']]
output_folder = Path.cwd()/'AFM_accepted_curves_CSV'
output_folder.mkdir(exist_ok=True)
tables = {}
for condition in CONDITIONS:
    included = sorted((c for c in accepted if c['condition']==condition),
                      key=lambda c:(c['cell'],c['crop'],int(c['curve']),int(c['position'])))
    rows = [HEADERS]
    last_crop = None
    for c in included:
        current = (c['cell'],c['crop'])
        if last_crop is not None and current != last_crop:
            rows.append([''] * len(HEADERS)) # Exactly one blank row between crops
        rows.append(export_row(c))
        last_crop = current
    tables[condition] = rows
    with (output_folder/CSV_NAMES[condition]).open('w',encoding='utf-8-sig',newline='') as handle:
        csv.writer(handle).writerows(rows)

with zipfile.ZipFile(Path.cwd()/'AFM_accepted_curves_CSV.zip','w',compression=zipfile.ZIP_DEFLATED) as archive:
    for condition in CONDITIONS:
        archive.write(output_folder/CSV_NAMES[condition],CSV_NAMES[condition])

# Optional four-sheet Excel output. The CSVs are always generated.
try:
    from artifact_tool import Workbook, SpreadsheetFile
    wb = Workbook.create()
    for condition in CONDITIONS:
        sheet = wb.worksheets.add(SHEET_NAMES[condition])
        matrix = tables[condition]
        numeric_cols = set(range(1,3)) | set(range(4,6)) | set(range(6,24))
        excel_rows = [[cell if row_index == 0 or col not in numeric_cols or cell == ''
                       else float(cell) for col,cell in enumerate(row)]
                      for row_index,row in enumerate(matrix)]
        # Allow literal blank rows between crop groups.
        sheet.get_range_by_indexes(0,0,len(excel_rows),len(HEADERS)).values = excel_rows
        sheet.get_range_by_indexes(0,0,1,len(HEADERS)).format = {
            'fill':'#14324C','font':{'bold':True,'color':'#FFFFFF','size':10},
            'wrap_text':True,'row_height':46,'vertical_alignment':'center'}
        sheet.freeze_panes.freeze_rows(1)
        sheet.get_range_by_indexes(0,0,len(excel_rows),len(HEADERS)).format.column_width = 19
        sheet.get_range_by_indexes(0,3,len(excel_rows),1).format.column_width = 26
        sheet.get_range_by_indexes(0,24,len(excel_rows),1).format.column_width = 40
    path = Path.cwd()/'AFM_accepted_curves.xlsx'
    SpreadsheetFile.export_xlsx(wb).save(str(path))
    print('Excel workbook:',path)
except ImportError:
    print('Excel workbook unavailable here. Four full-precision CSVs and a ZIP are ready.')

print('Accepted:',len(accepted),'of',len(curves),'individual curve files')
counts = Counter((c['condition'],c['cell']) for c in accepted)
for condition in CONDITIONS:
    print(condition,[(cell,counts[condition,cell]) for cell in (1,2,3)])
print('CSV folder:',output_folder)
print('ZIP:',Path.cwd()/'AFM_accepted_curves_CSV.zip')
for c in accepted:
    delta = c['values'].get('elasticity_rms_difference_pN')
    if delta is not None and abs(delta) > 0.01:
        print('RMS discrepancy: ',c['folder'],'Curve',c['curve'],
              'custom minus JPK =',round(delta,6),'pN')


Excel workbook unavailable here. Four full-precision CSVs and a ZIP are ready.
Accepted: 612 of 612 individual curve files
Live [(1, 50), (2, 52), (3, 51)]
20 mg/L [(1, 50), (2, 50), (3, 50)]
320 mg/L [(1, 50), (2, 51), (3, 53)]
Dead [(1, 53), (2, 51), (3, 51)]
CSV folder: /workspaces/AFM-plots_Jython_exports/AFM_accepted_curves_CSV
ZIP: /workspaces/AFM-plots_Jython_exports/AFM_accepted_curves_CSV.zip
RMS discrepancy:  Dead_cell1_crop2 Curve 126 custom minus JPK = -0.529936 pN
RMS discrepancy:  Dead_cell2_crop1 Curve 181 custom minus JPK = -0.595146 pN
RMS discrepancy:  Dead_cell2_crop1 Curve 99 custom minus JPK = -0.723249 pN
RMS discrepancy:  Dead_cell3_crop1 Curve 5 custom minus JPK = -0.053981 pN
